Libraries

In [ ]:
import pandas as pd

Google Mount

In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive mounted')
else:
    print('❌ Mount failed — run this cell again')

Mounted at /content/drive
✅ Drive mounted


Commit

In [6]:
# # ── CELL 1: Setup — run this ONCE per Colab session ─────────────────────────
# import subprocess
# import os
# import shutil
# from google.colab import userdata

# PAT = userdata.get('GITHUB_PAT')
# REPO_URL = f'https://{PAT}@github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation.git'
# REPO = '/content/project'
# BRANCH = 'btt_setup_VD'


# def setup_repo():
#     """Clone the repo fresh if it doesn't exist yet, otherwise just pull
#     the latest changes for my branch."""
#     if not os.path.exists(f'{REPO}/.git'):
#         # not cloned yet this session (or it's broken) — clone fresh
#         if os.path.exists(REPO):
#             print("Removing broken non-git folder...")
#             subprocess.run(['rm', '-rf', REPO])
#         print(f"Cloning branch '{BRANCH}' ...")
#         r = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO],
#                             capture_output=True, text=True)
#         print(r.stdout, r.stderr)
#     else:
#         print(f"Repo already exists — pulling latest '{BRANCH}' ...")
#         subprocess.run(['git', 'checkout', BRANCH], cwd=REPO)
#         r = subprocess.run(['git', 'pull', 'origin', BRANCH],
#                             capture_output=True, text=True, cwd=REPO)
#         print(r.stdout, r.stderr)

#     print("Is git repo:", os.path.exists(f'{REPO}/.git'))

#     # set git identity once (safe to re-run)
#     subprocess.run(['git', 'config', '--global', 'user.email', 'vd35@rice.edu'], cwd=REPO)
#     subprocess.run(['git', 'config', '--global', 'user.name', 'vantastics'], cwd=REPO)


# def detect_code_dir():
#     """Some branches put main.py/configs at repo root, others under code/.
#     Auto-detect which one this branch uses, same logic as 00_session_setup.ipynb."""
#     if os.path.exists(f'{REPO}/main.py'):
#         return REPO
#     elif os.path.exists(f'{REPO}/code/main.py'):
#         return f'{REPO}/code'
#     else:
#         raise FileNotFoundError(
#             f"Could not find main.py at {REPO}/main.py or {REPO}/code/main.py — "
#             f"check repo contents: {os.listdir(REPO)}"
#         )


# setup_repo()
# CODE_DIR = detect_code_dir()
# print(f"Code directory: {CODE_DIR}")

# # make my personal config copy — only if it doesn't already exist,
# # so re-running this cell never overwrites my own edits
# personal_config = f'{CODE_DIR}/configs/phase1_config_vd.yaml'
# shared_config = f'{CODE_DIR}/configs/phase1_config.yaml'
# if not os.path.exists(personal_config):
#     shutil.copy(shared_config, personal_config)
#     print(f"Created your personal config: {personal_config}")
# else:
#     print(f"Personal config already exists, leaving it as-is: {personal_config}")

Repo already exists — pulling latest 'btt_setup_VD' ...
Already up to date.
 From https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation
 * branch            btt_setup_VD -> FETCH_HEAD

Is git repo: True
Code directory: /content/project/code
Created your personal config: /content/project/code/configs/phase1_config_vd.yaml


In [ ]:
# ── CELL 2: Reusable commit helper — run setup_repo() first, then call this
#            any time I want to push a file to my branch.
import subprocess
import os
import shutil


def commit_file(source_path: str, dest_relative_path: str, commit_message: str,
                 under_code_dir: bool = True):
    """
    Copy a file into the repo and push it to your branch.

    source_path         — full path to the file right now (e.g. in Drive)
    dest_relative_path  — where it should live inside the repo (or code dir),
                           e.g. 'notebooks/fine_tuning_VD.ipynb'
    commit_message       — your commit message
    under_code_dir       — True if this path is relative to CODE_DIR (e.g.
                           notebooks/, configs/ — most things). False if it's
                           relative to the REPO root instead.
    """
    base_dir = CODE_DIR if under_code_dir else REPO
    dest_path = f'{base_dir}/{dest_relative_path}'
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    shutil.copy(source_path, dest_path)
    print(f"Copied to: {dest_path}")

    # git commands always run relative to REPO root, so the path passed to
    # `git add` needs to include the code/ prefix if that's where the file is
    git_relative_path = os.path.relpath(dest_path, REPO)

    for cmd in [
        ['git', 'checkout', BRANCH],
        ['git', 'add', git_relative_path],
        ['git', 'commit', '-m', commit_message],
        ['git', 'push', 'origin', BRANCH],
    ]:
        r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
        out = (r.stdout + r.stderr).strip()
        if out:
            print(out)

    print('\nDone. Verify at:')
    print(f'https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation/tree/{BRANCH}/{os.path.dirname(git_relative_path)}')


# ── Example usage ────────────────────────────────────────────────────────
# Every time you want to save a notebook (or any file) to your branch,
# just call this one line — no need to repeat the clone/copy/commit steps:

commit_file(
    source_path='/content/drive/MyDrive/Colab Notebooks/fine_tuning_VD.ipynb',
    dest_relative_path='notebooks/fine_tuning_VD.ipynb',
    commit_message='Update fine tuning notebook',
    # under_code_dir=True by default — since notebooks/ lives under code/
    # (confirm this matches your repo structure once you check CODE_DIR above)
)